
# VGGT 3D Volume Registration & Splatting Inference

This notebook performs full 3D volume reconstruction using a trained VGGT model.

**How it works:**
1. **Chunked Inference:** Because processing an entire MRI sequence (e.g., 70 slices) simultaneously would cause a GPU Out-Of-Memory (OOM) error, we split the slices into smaller chunks (e.g., 5 slices at a time).
2. **Coordinate Un-normalization:** The model outputs continuous 3D coordinates `[-1, 1]`. We convert these back to actual integer voxel indices `[0, W-1]`, `[0, H-1]`, `[0, Z-1]`.
3. **Splatting (Scatter-Add):** We take the pixel intensities from the moving image and "throw" them into a blank 3D volume at the coordinates predicted by the model. 
4. **Accumulation:** If two pixels land in the exact same voxel, we average their values to prevent flickering and ensure a smooth volume.


In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import cv2
from omegaconf import OmegaConf

# Ensure the VGGT modules can be imported
sys.path.append(os.path.abspath("."))

from vggt.models.vggt import VGGT
from training.data.datasets.mri_dataset import MRIDataset
from training.data.composed_dataset import ComposedDataset

In [ ]:
# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# Path to the specific subject/dataset folder
DATA_ROOT = "/home/minsukc/vggt/scratch/nifti/cardiac_1sec_10phase/nifti/card_3.125x3.125x3.125mm_256x256x70x10x4_snr20_fa60_bh"

SPLIT = "val"  # The dataset split to use
MODE = "dynamic"  # 'dynamic' because we are registering moving tissue
MRI_MODE = "axial"  # The slice orientation
TARGET_SIZE = 518  # The resolution required by the VGGT Vision Transformer

# Provide the path to your best trained model checkpoint.
# If None, it will use randomly initialized weights (good for testing the pipeline).
CHECKPOINT_PATH = None

# The maximum number of slices to process at once on the GPU.
# If you get a CUDA OOM error, reduce this number (e.g., to 2 or 3).
CHUNK_SIZE = 5

# Common config dict expected by the Dataset classes
common_conf = OmegaConf.create(
    {
        "img_size": TARGET_SIZE,
        "patch_size": 14,
        "augs": {"scales": [1.0, 1.0], "cojitter": False, "cojitter_ratio": 0.5, "color_jitter": None, "gray_scale": False, "gau_blur": False},
        "rescale": False,
        "rescale_aug": False,
        "landscape_check": False,
        "debug": False,
    }
)

In [ ]:
# =============================================================================
# 2. INITIALIZE AND LOAD MODEL
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize the model exactly as it was configured during training.
# Note: use_z_pose_embedding=True matches our latest implementation for axial slices.
model = VGGT(
    img_size=TARGET_SIZE,
    patch_size=14,
    embed_dim=1024,
    enable_camera=False,
    enable_point=True,  # We only need the point head for registration
    enable_depth=False,
    enable_track=False,
    use_z_pose_embedding=True,
)

if CHECKPOINT_PATH is not None and os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")

    # Handle different checkpoint formatting styles (PyTorch Lightning vs raw)
    if "model" in ckpt:
        model.load_state_dict(ckpt["model"], strict=False)
    elif "state_dict" in ckpt:
        model.load_state_dict(ckpt["state_dict"], strict=False)
    else:
        model.load_state_dict(ckpt, strict=False)
else:
    print("WARNING: No valid checkpoint provided. Model will output random noise.")

model.to(device)
model.eval()  # Set to evaluation mode to disable dropout, etc.
print("Model ready.")

In [ ]:
# =============================================================================
# 3. SPLATTING ALGORITHM
# =============================================================================
def splat_to_volume(images, world_points, masks, target_shape, volume_sum=None, volume_count=None):
    """
    Maps 2D image pixels into a 3D voxel grid based on predicted continuous coordinates.

    Args:
        images: (S, C, H, W) Tensor of image chunks [0, 1]
        world_points: (S, H, W, 3) Tensor of predicted coordinates [-1, 1]
        masks: (S, H, W) Boolean tensor indicating valid tissue (ignores black padding)
        target_shape: Tuple (W, H, Z) of the final 3D volume
        volume_sum: 3D numpy array accumulating pixel colors.
        volume_count: 3D numpy array counting how many pixels landed in each voxel.

    Returns:
        Updated volume_sum and volume_count.
    """
    W, H, Z = target_shape
    C = images.shape[1]

    # Initialize CPU accumulators if this is the first chunk
    if volume_sum is None:
        volume_sum = np.zeros((C, W, H, Z), dtype=np.float32)
        volume_count = np.zeros((W, H, Z), dtype=np.float32)

    max_dim = max(W, H, Z) - 1
    scale_factor = max_dim / 2.0

    # Process slice by slice within the current chunk
    for i in range(images.shape[0]):
        img = images[i].detach().cpu().numpy()
        wp = world_points[i].detach().cpu().numpy()
        mask = masks[i].detach().cpu().numpy()

        # Apply mask: Filter out the black padding added by the dataset
        valid_img = img[:, mask]  # Shape: (C, num_valid_pixels)
        valid_wp = wp[mask]  # Shape: (num_valid_pixels, 3)

        if valid_wp.shape[0] == 0:
            continue

        # Step A: Un-normalize coordinates from [-1, 1] to voxel indices [0, W-1]
        x_coords = valid_wp[:, 0] * scale_factor + (W - 1) / 2.0
        y_coords = valid_wp[:, 1] * scale_factor + (H - 1) / 2.0
        z_coords = valid_wp[:, 2] * scale_factor + (Z - 1) / 2.0

        # Step B: Round floating point coordinates to nearest integer voxel index
        x_idx = np.clip(np.round(x_coords), 0, W - 1).astype(np.int32)
        y_idx = np.clip(np.round(y_coords), 0, H - 1).astype(np.int32)
        z_idx = np.clip(np.round(z_coords), 0, Z - 1).astype(np.int32)

        # Step C: Scatter-Add (Splatting)
        # We add the pixel intensity to the specific voxel it landed in.
        # np.add.at is crucial here because multiple pixels might land in the same voxel,
        # and standard indexing (volume[x]=y) would just overwrite them instead of adding.
        for c in range(C):
            np.add.at(volume_sum[c], (x_idx, y_idx, z_idx), valid_img[c])

        # Keep track of how many pixels landed in this voxel so we can average them later
        np.add.at(volume_count, (x_idx, y_idx, z_idx), 1.0)

    return volume_sum, volume_count


In [ ]:
# =============================================================================
# 4. CHUNKED INFERENCE & RECONSTRUCTION
# =============================================================================

# Initialize Dataset
print("Loading Dataset...")
dataset = MRIDataset(common_conf, DATA_ROOT, split=SPLIT, mode=MODE, num_slices=70, target_size=TARGET_SIZE, mri_mode=MRI_MODE)

# Use ComposedDataset to apply standard data formatting and normalization
comp_dataset = ComposedDataset({"target": "dummy"}, common_conf)
comp_dataset.datasets = [dataset]
comp_dataset.cumulative_sizes = [len(dataset)]
comp_dataset.total_samples = len(dataset)

# Fetch all slices for sequence 0 at a random time 't'
print(f"Fetching full sequence...")
raw_batch = dataset.get_data(seq_index=0, img_per_seq=70)
batch = comp_dataset.__getitem__((0, 0, raw_batch))

total_slices = batch["images"].shape[0]
print(f"Total slices to process: {total_slices}")

# Extract original volume shape (before 518x518 padding) from metadata
W = int(batch["original_sizes"][0, 1].item())
H = int(batch["original_sizes"][0, 0].item())
Z = total_slices
target_shape = (W, H, Z)
print(f"Target Volume Shape: {target_shape}")

# Initialize Accumulators
volume_sum = None
volume_count = None
all_pred_wp = []

print("Starting chunked inference...")
with torch.no_grad():
    for i in range(0, total_slices, CHUNK_SIZE):
        chunk_end = min(i + CHUNK_SIZE, total_slices)
        print(f"  Processing chunk {i} to {chunk_end - 1}...")

        # 1. Slice the batch arrays to get only the current chunk of slices
        chunk_images = batch["images"][i:chunk_end].unsqueeze(0).to(device)  # Shape: (1, S, C, H, W)
        chunk_masks = batch["point_masks"][i:chunk_end]  # Shape: (S, H, W)
        chunk_batch = {"z_indices": batch["z_indices"][i:chunk_end].unsqueeze(0).to(device), "scanner_coords": batch["scanner_coords"][i:chunk_end].unsqueeze(0).to(device), "scale_factors": batch["scale_factors"][i:chunk_end].unsqueeze(0).to(device)}

        # 2. Forward pass through the Transformer
        predictions = model(images=chunk_images, batch=chunk_batch)

        # 3. Extract the predicted 3D coordinates
        pred_wp = predictions["world_points"].squeeze(0)  # Shape: (S, H, W, 3)
        images_squeezed = chunk_images.squeeze(0)  # Shape: (S, C, H, W)

        # 4. Splat this chunk's predictions into our CPU accumulators
        all_pred_wp.append(pred_wp.cpu())
        volume_sum, volume_count = splat_to_volume(images_squeezed, pred_wp, chunk_masks, target_shape=target_shape, volume_sum=volume_sum, volume_count=volume_count)

# Finalize Volume: Average overlapping pixels
valid_mask = volume_count > 0
registered_volume = np.zeros_like(volume_sum)

for c in range(registered_volume.shape[0]):
    # Only divide by count where count > 0 to prevent divide-by-zero NaN errors
    registered_volume[c][valid_mask] = volume_sum[c][valid_mask] / volume_count[valid_mask]

all_pred_wp = torch.cat(all_pred_wp, dim=0)
print("\nInference and Splatting Complete!")


In [ ]:
# =============================================================================
# 5. VISUALIZE RESULTS (REGISTRATION ERROR)
# =============================================================================
num_vis = 5
z_indices_to_show = np.linspace(0, Z - 1, num_vis, dtype=int)

fig, axes = plt.subplots(3, num_vis, figsize=(20, 12))
fig.suptitle(f"Registered Volume Comparison & Error", fontsize=16)

gray_volume = registered_volume.mean(axis=0)
sub_dir = dataset.subjects[0]
gt_frame1 = nib.load(os.path.join(sub_dir, "img_t001.nii.gz")).get_fdata()
# Normalize GT to [0, 1] for comparison
gt_frame1 = (gt_frame1 - gt_frame1.min()) / (gt_frame1.max() - gt_frame1.min() + 1e-8)

for i, z_idx in enumerate(z_indices_to_show):
    # 1. GT Reference
    axes[0, i].imshow(gt_frame1[:, :, z_idx].T, cmap="gray", origin="lower")
    axes[0, i].set_title(f"GT Frame 1 (z={z_idx})")
    axes[0, i].axis("off")

    # 2. Predicted Registered
    axes[1, i].imshow(gray_volume[:, :, z_idx].T, cmap="gray", origin="lower")
    axes[1, i].set_title(f"Pred Registered (z={z_idx})")
    axes[1, i].axis("off")

    # 3. Difference Map (Absolute Error)
    diff = np.abs(gray_volume[:, :, z_idx] - gt_frame1[:, :, z_idx])
    im = axes[2, i].imshow(diff.T, cmap="hot", origin="lower")
    axes[2, i].set_title(f"Abs Difference (z={z_idx})")
    axes[2, i].axis("off")
    fig.colorbar(im, ax=axes[2, i])

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 6. DVF ERROR ANALYSIS (GT DVF vs PRED DVF)
# =============================================================================
H_tgt, W_tgt = TARGET_SIZE, TARGET_SIZE
max_dim = max(W, H, Z) - 1
scale_factor = max_dim / 2.0

# 1. Reconstruct Flat Scanner Coordinates (Static Units)
scanner_coords = []
for i in range(total_slices):
    r, c = np.meshgrid(np.arange(W), np.arange(H), indexing="ij")
    coords = np.stack([r, c, np.full_like(r, i)], axis=-1).astype(np.float32)
    coords[..., 0] = (coords[..., 0] - (W - 1) / 2) / scale_factor
    coords[..., 1] = (coords[..., 1] - (H - 1) / 2) / scale_factor
    coords[..., 2] = (coords[..., 2] - (Z - 1) / 2) / scale_factor
    
    sc = TARGET_SIZE / max(H, W)
    nw, nh = int(W * sc), int(H * sc)
    coords_r = cv2.resize(coords, (nw, nh), interpolation=cv2.INTER_LINEAR)
    pt, pl = (TARGET_SIZE - nh) // 2, (TARGET_SIZE - nw) // 2
    coords_p = np.pad(coords_r, ((pt, TARGET_SIZE - nh - pt), (pl, TARGET_SIZE - nw - pl), (0, 0)), constant_values=-2.0)
    scanner_coords.append(coords_p)

scanner_coords = torch.from_numpy(np.stack(scanner_coords))
pred_dvf = scanner_coords - all_pred_wp
gt_dvf = scanner_coords - batch["world_points"]

fig, axes = plt.subplots(2, num_vis, figsize=(20, 8))
fig.suptitle("DVF Magnitude: GT vs Prediction", fontsize=16)
masks = batch["point_masks"]

for i, z_idx in enumerate(z_indices_to_show):
    gt_mag = torch.norm(gt_dvf[z_idx], dim=-1) * scale_factor
    pred_mag = torch.norm(pred_dvf[z_idx], dim=-1) * scale_factor
    gt_mag[~masks[z_idx]] = 0; pred_mag[~masks[z_idx]] = 0

    im1 = axes[0, i].imshow(gt_mag.T, cmap="jet", origin="lower")
    axes[0, i].set_title(f"GT DVF Mag (z={z_idx})")
    fig.colorbar(im1, ax=axes[0, i])
    im2 = axes[1, i].imshow(pred_mag.T, cmap="jet", origin="lower")
    axes[1, i].set_title(f"Pred DVF Mag (z={z_idx})")
    fig.colorbar(im2, ax=axes[1, i])

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 7. QUANTITATIVE METRICS
# =============================================================================
vol_mask = (gt_frame1 > 0) & (gray_volume > 0)
vol_mae = np.mean(np.abs(gray_volume[vol_mask] - gt_frame1[vol_mask]))

pred_dvf_masked = pred_dvf[masks]
gt_dvf_masked = gt_dvf[masks]
dvf_error_voxels = (pred_dvf_masked - gt_dvf_masked) * scale_factor
dvf_mae = torch.abs(dvf_error_voxels).mean().item()

print("--- Quantitative Results ---")
print(f"Volume Registration MAE: {vol_mae:.4f} (intensity)")
print(f"DVF Prediction MAE:      {dvf_mae:.4f} voxels")

In [ ]:
# =============================================================================
# 6. SAVE AS NIFTI (.nii.gz)
# =============================================================================
SAVE_PATH = "registered_output.nii.gz"

# Create a Nifti1Image object.
# We pass the constructed gray_volume (W, H, Z) and an identity affine matrix.
# If you have the original affine matrix from the raw dataset, you can replace np.eye(4) with it.
nifti_img = nib.Nifti1Image(gray_volume, affine=np.eye(4))

# Save to disk
nib.save(nifti_img, SAVE_PATH)
print(f"Successfully saved registered volume to: {SAVE_PATH}")
